# Biomimicry #

In [ ]:
%pip install -q google-genai
%pip install scikit-learn
%pip install scikit-learn pandas


In [ ]:
BIOMIMICRY_DB = [
    {
        "id": "B01",
        "organism": "Jajko kurze",
        "mechanism": "Skorupka warstwowa z mikroporami",
        "function": "cienka, sztywna powłoka chroni delikatną zawartość przed uderzeniami "
                    "i naciskiem, a mikroskopijne pory umożliwiają wymianę gazową i naturalny rozkład",
        "principle": "wytrzymałość przez geometrię kopuły, nie grubość materiału",
    },
    {
        "id": "B02",
        "organism": "Grzybnia (mycelium)",
        "mechanism": "Sieć strzępek grzybowych spajająca odpady organiczne",
        "function": "tworzy sztywną, lekką strukturę ochronną z odpadów rolniczych, "
                    "w pełni kompostowalną w ciągu tygodni",
        "principle": "wzrost struktury zamiast produkcji materiału, pełna biodegradacja",
    },
    {
        "id": "B03",
        "organism": "Skórka pomarańczy",
        "mechanism": "Wielowarstwowa, elastyczna okrywa z gąbczastą warstwą wewnętrzną",
        "function": "amortyzuje uderzenia mechaniczne, chroni przed utratą wilgoci, "
                    "w pełni jadalna i biodegradowalna",
        "principle": "amortyzacja przez warstwę porowatą, ochrona przed wysychaniem",
    },
    {
        "id": "B04",
        "organism": "Muszla mięczaka (macica perłowa / nacre)",
        "mechanism": "Naprzemienne mikrowarstwy węglanu wapnia i białka",
        "function": "bardzo wysoka wytrzymałość na uderzenia przy minimalnej ilości materiału, "
                    "pęknięcia nie propagują się przez całą strukturę",
        "principle": "warstwowa architektura zatrzymuje pęknięcia, wytrzymałość bez grubości",
    },
    {
        "id": "B05",
        "organism": "Sieć pajęcza",
        "mechanism": "Włókno białkowe o wysokiej wytrzymałości na rozciąganie",
        "function": "duża wytrzymałość przy bardzo małej masie materiału, w pełni biodegradowalne",
        "principle": "maksymalna wytrzymałość na jednostkę masy materiału",
    },
    {
        "id": "B06",
        "organism": "Szyszka sosnowa",
        "mechanism": "Łuski reagujące na wilgotność (otwierają/zamykają się)",
        "function": "struktura zmienia kształt i przepuszczalność w reakcji na wilgoć, "
                    "bez żadnych ruchomych części mechanicznych",
        "principle": "aktuacja pasywna sterowana wilgotnością, zero energii z zewnątrz",
    },
    {
        "id": "B07",
        "organism": "Liść lotosu",
        "mechanism": "Mikro- i nanostruktura powierzchni odpychająca wodę i brud",
        "function": "powierzchnia samoczyszcząca się, odporna na wilgoć bez powłok chemicznych",
        "principle": "właściwości hydrofobowe przez mikrostrukturę, nie przez powłokę chemiczną",
    },
    {
        "id": "B08",
        "organism": "Struktura plastra miodu (pszczoły)",
        "mechanism": "Regularna siatka sześciokątnych komórek",
        "function": "maksymalna wytrzymałość konstrukcyjna przy minimalnym zużyciu materiału",
        "principle": "geometria heksagonalna optymalizuje relację wytrzymałość/materiał",
    },
    {
        "id": "B09",
        "organism": "Skóra jeżozwierza / kolce jeża",
        "mechanism": "Sztywne wypustki rozpraszające punktowy nacisk",
        "function": "rozprasza siłę uderzenia na większą powierzchnię, chroniąc rdzeń",
        "principle": "rozpraszanie sił punktowych przez strukturę powierzchniową",
    },
    {
        "id": "B10",
        "organism": "Nasiona roślin (np. mniszek, klon)",
        "mechanism": "Lekka struktura włóknista transportowana przez wiatr",
        "function": "ochrona zarodka przy minimalnej masie materiału opakowującego",
        "principle": "minimalizacja masy materiału ochronnego względem chronionej zawartości",
    },
]


In [ ]:
DEFAULT_PROBLEM = (
    "Jablka niszcza sie podczas transportu i przechowywania z powodu uderzen "
    "i utraty wilgoci, co obniza sprzedaz. Potrzebujemy opakowania, ktore skutecznie "
    "chroni owoce przed wstrzasami i wysychaniem, a po dostarczeniu produktu "
    "bezpiecznie i szybko sie rozklada lub jest ponownie wykorzystane."
)

DEFAULT_FUNCTION_QUERY = (
    "ochrona przed uderzeniami mechanicznymi i utrata wilgoci, "
    "lekka struktura, pelna biodegradacja lub kompostowalnosc po uzyciu"
)

wpisany_problem = input("Opisz problem (Enter = przyklad z jablkami): ").strip()
PROBLEM_STATEMENT = wpisany_problem if wpisany_problem else DEFAULT_PROBLEM

wpisane_zapytanie = input(
    "Opisz funkcje, ktorej szuka rozwiazanie (Enter = domyslne zapytanie): "
).strip()
FUNCTION_QUERY = wpisane_zapytanie if wpisane_zapytanie else DEFAULT_FUNCTION_QUERY

print()
print("Problem:", PROBLEM_STATEMENT)
print()
print("Zapytanie funkcjonalne:", FUNCTION_QUERY)

In [ ]:
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

corpus = [entry["function"] for entry in BIOMIMICRY_DB] + [FUNCTION_QUERY]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

query_vector = tfidf_matrix[-1]
db_vectors = tfidf_matrix[:-1]

similarities = cosine_similarity(query_vector, db_vectors).flatten()

ranking = pd.DataFrame({
    "id": [e["id"] for e in BIOMIMICRY_DB],
    "organism": [e["organism"] for e in BIOMIMICRY_DB],
    "mechanism": [e["mechanism"] for e in BIOMIMICRY_DB],
    "similarity": similarities,
}).sort_values("similarity", ascending=False).reset_index(drop=True)

In [ ]:
TOP_N = 3
top_matches = ranking.head(TOP_N)
selected_ids = top_matches["id"].tolist()
selected_entries = [e for e in BIOMIMICRY_DB if e["id"] in selected_ids]

for e in selected_entries:
    print(f"[{e['id']}] {e['organism']} -- {e['mechanism']}")

In [ ]:
import os
import json

def generate_concept_from_mechanism(problem, entry):
    """Zamienia mechanizm biologiczny na konkretny koncept opakowania.

    Zwraca dict z polami: id, tytul, opis, zrodlo_mechanizmu -- gotowy do dalszej
    ewaluacji w module oceny kandydatow (wspolnym dla TRIZ i biomimikry).

    Uzywa Google Gemini API (pakiet "google-genai"). Wymaga zmiennej
    srodowiskowej GEMINI_API_KEY lub GOOGLE_API_KEY.
    """
    prompt = f"""Jestes inzynierem projektujacym ekologiczne opakowania.

Problem: {problem}

Mechanizm z natury do wykorzystania jako inspiracja:
- Organizm: {entry["organism"]}
- Mechanizm: {entry["mechanism"]}
- Funkcja: {entry["function"]}
- Zasada dzialania: {entry["principle"]}

Zaproponuj JEDEN konkretny, wdrozalny koncept opakowania inspirowany tym mechanizmem.
Odpowiedz WYLACZNIE czystym JSON (bez markdown, bez ```), z polami:
tytul (krotki, 3-6 slow) i opis (2-3 zdania, konkretnie jak dziala i z jakiego materialu)."""

    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

    if api_key:
        from google import genai
        from google.genai import types

        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
            ),
        )
        parsed = json.loads(response.text)
    else:
        parsed = {
            "tytul": "Koncept inspirowany: " + entry["organism"],
            "opis": (
                "[PLACEHOLDER -- brak GEMINI_API_KEY] Rozwiniecie mechanizmu "
                + entry["mechanism"] + " (" + entry["principle"]
                + ") w konkretny projekt opakowania dla podanego problemu."
            ),
        }

    return {
        "id": entry["id"],
        "zrodlo_mechanizmu": entry["organism"] + " -- " + entry["mechanism"],
        "tytul": parsed["tytul"],
        "opis": parsed["opis"],
    }

In [ ]:
candidates = [
    generate_concept_from_mechanism(PROBLEM_STATEMENT, entry)
    for entry in selected_entries
]

for c in candidates:
    print("[" + c["id"] + "] " + c["tytul"])
    print("  zrodlo: " + c["zrodlo_mechanizmu"])
    print("  opis:   " + c["opis"])
    print()

In [ ]:
reasoning_trail = {
    "method": "biomimicry",
    "problem": PROBLEM_STATEMENT,
    "function_query": FUNCTION_QUERY,
    "similarity_ranking": ranking.to_dict(orient="records"),
    "selected_mechanisms": selected_ids,
    "candidates": candidates,
}

print(json.dumps(reasoning_trail, indent=2, ensure_ascii=False))